<a href="https://colab.research.google.com/github/wr2401/codelearning/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
from cnn_models import LeNet5, compute_accuracy

In [21]:
transform = transforms.Compose([
    transforms.Resize((28,28)),          #调整图像大小为28×28
    transforms.ToTensor(),            #转换为Tensor
])

In [22]:
train_dataset=datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset=datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [23]:
#使用Dataloader加载训练数据
train_loader=DataLoader(
    train_dataset, #传入训练数据集
    batch_size=32, #批次数量
    shuffle=True, #打乱数据
    num_workers=8 #加载数据进程数
)

test_loader=DataLoader(
    test_dataset, #传入测试数据集
    batch_size=32, #每批次数量
    shuffle=False, #不打乱数据
    num_workers=8 #加载数据进程数
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [24]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [25]:
model=LeNet5().to(device) #初始化模型，并将模型移至GPU
criterion=nn.CrossEntropyLoss() #分类任务使用交叉熵损失
optimizer=optim.Adam(model.parameters(),lr=0.001) #设置优化器

In [32]:
num_epochs=7
for epoch in range(num_epochs):
  model.train()
  total_loss=0
  for inputs,labels in train_loader:
    inputs,labels=inputs.to(device),labels.to(device)
    optimizer.zero_grad()
    outputs=model(inputs)
    loss=criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"Epoch[{epoch+1}/{num_epochs}],Loss:{total_loss/len(train_loader):.4f}")
  train_accuracy=compute_accuracy(train_loader,device, model)
  test_accuracy=compute_accuracy(test_loader,device, model)
  print(f"Train Accuracy:{train_accuracy:.2f}%,Test Accuracy:{test_accuracy:.2f}%")

Epoch[1/7],Loss:0.0035
Train Accuracy:99.90%,Test Accuracy:98.97%
Epoch[2/7],Loss:0.0045
Train Accuracy:99.86%,Test Accuracy:99.00%
Epoch[3/7],Loss:0.0025
Train Accuracy:99.97%,Test Accuracy:99.09%
Epoch[4/7],Loss:0.0044
Train Accuracy:99.97%,Test Accuracy:99.09%
Epoch[5/7],Loss:0.0029
Train Accuracy:99.93%,Test Accuracy:98.94%
Epoch[6/7],Loss:0.0039
Train Accuracy:99.92%,Test Accuracy:99.01%
Epoch[7/7],Loss:0.0034
Train Accuracy:99.97%,Test Accuracy:99.10%


In [27]:
%%writefile cnn_models.py

import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1=nn.Conv2d(1,6,kernel_size=5,padding=2)
    self.pool1=nn.AvgPool2d(kernel_size=2)
    self.conv2=nn.Conv2d(6,16,kernel_size=5)
    self.pool2=nn.AvgPool2d(kernel_size=2)
    self.conv3=nn.Conv2d(16,120,kernel_size=5)
    self.flatten=nn.Flatten()
    self.fc1=nn.Linear(120,84)
    self.fc2=nn.Linear(84,10)
  def forward(self,x):
    x = F.relu(self.conv1(x))
    x = self.pool1(x)
    x = F.relu(self.conv2(x))
    x = self.pool2(x)
    x = F.relu(self.conv3(x))
    x = self.flatten(x)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

def compute_accuracy(loader, device, model):
  correct=0
  total=0
  with torch.no_grad():
    for inputs,labels in loader:
      inputs,labels=inputs.to(device),labels.to(device)
      outputs=model(inputs)
      max_values,predicted=torch.max(outputs,1)
      total+=labels.size(0)
      correct+=(predicted==labels).sum().item()
  accuracy=100*correct/total
  return accuracy


Overwriting cnn_models.py
